In [ ]:
!pip uninstall torchaudio -y

In [ ]:
!git clone https://github.com/Mohamedmagdy21/sentiment_analyzer.git

%cd sentiment_analyzer

!pip install -r requirements.txt
# Remove torchvision (causes nms operator crash with Kaggle's torch)
!pip uninstall torchvision -y

In [ ]:
import os
os.makedirs("Data/raw/twitter_raw", exist_ok=True)
!cp /kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv \
    Data/raw/twitter_raw/training.1600000.processed.noemoticon.csv
print("Copied raw Twitter data from Kaggle input.")

In [ ]:
!python -m preprocessing.preprocess dataset=twitter hydra.run.dir=.

In [ ]:
!HYDRA_FULL_ERROR=1 python -m training.train \
    dataset=twitter \
    model=twitter_roberta \
    hydra.run.dir=.

In [ ]:
import shutil, os, json
shutil.rmtree("Data", ignore_errors=True)
shutil.rmtree(".git", ignore_errors=True)
shutil.rmtree("artifacts/checkpoints", ignore_errors=True)
for f in os.listdir("."):
    if f.endswith(".csv") or f.endswith(".log"):
        os.remove(f)
print("Cleaned up large files from output.")

In [ ]:
import os, json, subprocess
model_dir = "artifacts/models/twitter"
os.makedirs(model_dir, exist_ok=True)
meta = {"id": "mohamedmagdyw/twitter-roberta", "title": "Twitter RoBERTa Sentiment Model", "licenses": [{"name": "CC0-1.0"}]}
with open(os.path.join(model_dir, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f)
r = subprocess.run(["kaggle", "datasets", "version", "-p", model_dir, "-m", f"v39 - {os.environ.get('KAGGLE_KERNEL_VERSION','?')}", "--dir-mode", "zip"], capture_output=True, text=True)
print(r.stdout[-500:] if len(r.stdout)>500 else r.stdout)
if r.returncode != 0:
    # Dataset may not exist yet, try create
    r2 = subprocess.run(["kaggle", "datasets", "create", "-p", model_dir, "--dir-mode", "zip"], capture_output=True, text=True)
    print(r2.stdout[-500:] if len(r2.stdout)>500 else r2.stdout)
    print(r2.stderr[:500] if r2.stderr else "")